# 09 — FIFA Ranking + XGBoost + Neural Network

Adds `home_fifa_points` / `away_fifa_points` as a 3rd ranking signal alongside Elo and squad value. Then compares three classifiers on the same feature set:

1. **Multinomial logistic regression** (current notebook-06 baseline)
2. **XGBoost** (non-linear, captures interactions)
3. **Sklearn MLPClassifier** (small 2-layer neural net; PyTorch is overkill for ~5k rows of tabular data)

Walk-forward backtest on WC 2010-2022.

In [1]:
import sys
sys.path.append('..')
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss
import xgboost as xgb
print('xgboost', xgb.__version__)

from src.elo import EloModel
from src.draw_model import match_outcome

xgboost 3.2.0


## Build feature set

Use the Fjelstul-overlay matches_with_squad_value.csv. Add FIFA points diff alongside elo + value + age + top1.

In [2]:
df_all = pd.read_csv('../data/processed/matches_competitive.csv', parse_dates=['date']).dropna(subset=['home_score','away_score'])
df_sv = pd.read_csv('../data/processed/matches_with_squad_value.csv', parse_dates=['date'])

elo = EloModel()
elo_enriched = elo.fit(df_all)
df = df_sv.merge(elo_enriched[['date','home_team','away_team','home_elo_pre','away_elo_pre']],
                  on=['date','home_team','away_team'], how='left').dropna(subset=['home_elo_pre','away_elo_pre'])

HOME_ADVANTAGE = 100
df['neutral'] = df['neutral'].fillna(False)
df['eff_elo_diff'] = (df['home_elo_pre'] + (~df['neutral']).astype(int) * HOME_ADVANTAGE) - df['away_elo_pre']
df['elo_diff_norm'] = df['eff_elo_diff'] / 400.0

FLOOR = 1e5
df['log_value_diff']    = np.log10(df['home_top_n_value_eur'].clip(lower=FLOOR) / df['away_top_n_value_eur'].clip(lower=FLOOR))
df['outfield_age_diff'] = df['home_outfield_mean_age'] - df['away_outfield_mean_age']
df['top1_share_diff']   = df['home_top1_share']        - df['away_top1_share']
df['fifa_points_diff']  = (df['home_fifa_points'] - df['away_fifa_points']) / 1000.0

df['outcome'] = df.apply(lambda r: match_outcome(r['home_score'], r['away_score']), axis=1)

FEATURES = ['elo_diff_norm','log_value_diff','outfield_age_diff','top1_share_diff','fifa_points_diff']
valid = df.dropna(subset=FEATURES + ['outcome'])
valid = valid[(valid['home_top_n_value_eur'] > FLOOR) & (valid['away_top_n_value_eur'] > FLOOR)].copy()
print(f'Training-eligible matches: {len(valid):,}')
print(valid[FEATURES].describe().round(3))

Training-eligible matches: 7,464
       elo_diff_norm  log_value_diff  outfield_age_diff  top1_share_diff  \
count       7464.000        7464.000           7464.000         7464.000   
mean           0.209           0.048             -0.050           -0.009   
std            0.437           1.103              2.625            0.288   
min           -1.601          -3.751            -12.233           -0.925   
25%           -0.075          -0.668             -1.584           -0.167   
50%            0.197           0.046             -0.050           -0.007   
75%            0.482           0.778              1.517            0.148   
max            2.077           3.840             12.078            0.926   

       fifa_points_diff  
count          7464.000  
mean              0.011  
std               0.356  
min              -1.775  
25%              -0.198  
50%               0.018  
75%               0.218  
max               1.669  


## Three models, walk-forward backtest

With and without FIFA points, to isolate ranking contribution.

In [3]:
def score(y_true, proba):
    p = np.clip(proba, 1e-6, 1 - 1e-6)
    # 3-class Brier: sum_c (p_c - 1{y=c})^2, averaged over matches. Range 0..2; lower = better.
    onehot = np.zeros_like(p)
    onehot[np.arange(len(y_true)), y_true] = 1
    brier = float(np.mean(np.sum((p - onehot) ** 2, axis=1)))
    return {
        'log_loss': log_loss(y_true, p, labels=[0, 1, 2]),
        'accuracy': float((p.argmax(axis=1) == y_true).mean()),
        'brier': brier,
    }

def fit_logit(X, y):
    clf = LogisticRegression(max_iter=2000)
    clf.fit(X, y)
    return clf

def fit_xgb(X, y):
    clf = xgb.XGBClassifier(
        objective='multi:softprob', num_class=3,
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        eval_metric='mlogloss', n_jobs=4, verbosity=0,
    )
    clf.fit(X, y)
    return clf

def fit_mlp(X_scaled, y):
    clf = MLPClassifier(hidden_layer_sizes=(32, 16), activation='relu',
                        max_iter=500, learning_rate_init=0.005,
                        alpha=1e-3, early_stopping=True, random_state=0)
    clf.fit(X_scaled, y)
    return clf

def predict_proba_ordered(clf, X):
    proba = clf.predict_proba(X)
    order = [list(clf.classes_).index(c) for c in (0, 1, 2)]
    return proba[:, order]

FEAT_NO_FIFA   = ['elo_diff_norm','log_value_diff','outfield_age_diff','top1_share_diff']
FEAT_WITH_FIFA = FEAT_NO_FIFA + ['fifa_points_diff']

rows = []
for year in [2010, 2014, 2018, 2022]:
    train = valid[valid['date'].dt.year < year]
    test  = valid[(valid['date'].dt.year == year) & (valid['tournament'] == 'FIFA World Cup')]
    if test.empty:
        continue
    y_train = train['outcome'].to_numpy()
    y_test = test['outcome'].to_numpy()
    row = {'year': year, 'n_train': len(train), 'n_test': len(test)}
    for feat_name, cols in [('no_fifa', FEAT_NO_FIFA), ('with_fifa', FEAT_WITH_FIFA)]:
        sc = StandardScaler().fit(train[cols].to_numpy())
        Xtr = train[cols].to_numpy(); Xte = test[cols].to_numpy()
        Xtr_s = sc.transform(Xtr); Xte_s = sc.transform(Xte)
        for model_name, clf in [
            ('logit', fit_logit(Xtr, y_train)),
            ('xgb',   fit_xgb(Xtr, y_train)),
            ('mlp',   fit_mlp(Xtr_s, y_train)),
        ]:
            X_pred = Xte_s if model_name == 'mlp' else Xte
            p = predict_proba_ordered(clf, X_pred)
            s = score(y_test, p)
            k = f'{model_name}_{feat_name}'
            row[f'{k}_ll']  = s['log_loss']
            row[f'{k}_acc'] = s['accuracy']
            row[f'{k}_br']  = s['brier']
    rows.append(row)

results = pd.DataFrame(rows)
ll_cols = [c for c in results.columns if c.endswith('_ll')]
acc_cols = [c for c in results.columns if c.endswith('_acc')]
print('Log loss per year:')
print(results[['year','n_train','n_test'] + ll_cols].round(4).to_string(index=False))
print()
print('Accuracy per year:')
print(results[['year'] + acc_cols].round(4).to_string(index=False))
print()
print('Averages:')
for name in ['logit_no_fifa','logit_with_fifa','xgb_no_fifa','xgb_with_fifa','mlp_no_fifa','mlp_with_fifa']:
    print(f"  {name:18s}  log_loss: {results[f'{name}_ll'].mean():.4f}   accuracy: {results[f'{name}_acc'].mean():.4f}   brier: {results[f'{name}_br'].mean():.4f}")

Log loss per year:
 year  n_train  n_test  logit_no_fifa_ll  xgb_no_fifa_ll  mlp_no_fifa_ll  logit_with_fifa_ll  xgb_with_fifa_ll  mlp_with_fifa_ll
 2010     1446      64            1.0215          0.9276          1.0327              1.0185            0.9389            1.0107
 2014     2747      64            0.9688          1.0035          0.9648              0.9698            0.9448            0.9558
 2018     4082      64            0.9989          0.9931          0.9945              0.9975            1.0442            0.9719
 2022     5646      64            1.0556          1.1296          1.0755              1.0528            1.1037            1.0642

Accuracy per year:
 year  logit_no_fifa_acc  xgb_no_fifa_acc  mlp_no_fifa_acc  logit_with_fifa_acc  xgb_with_fifa_acc  mlp_with_fifa_acc
 2010             0.5156           0.6094           0.4844               0.5156             0.5625             0.5156
 2014             0.5938           0.5000           0.5938               0.5938 

## XGBoost feature importance

In [4]:
last_train = valid[valid['date'].dt.year < 2022]
clf_xgb = fit_xgb(last_train[FEAT_WITH_FIFA].to_numpy(), last_train['outcome'].to_numpy())
imp = pd.Series(clf_xgb.feature_importances_, index=FEAT_WITH_FIFA).sort_values(ascending=False)
print('XGBoost gain-based importance:')
print(imp.round(3).to_string())

XGBoost gain-based importance:
elo_diff_norm        0.367
fifa_points_diff     0.228
log_value_diff       0.180
top1_share_diff      0.114
outfield_age_diff    0.110


## Add manager features and re-run

Manager features are WC-only (Fjelstul `manager_appearances`). For non-WC training matches both home and away `mgr_*` values are 0, so the model can only learn from the WC subset of training data — but the 256 WC matches in our walk-forward training are enough to estimate a coefficient since manager experience varies meaningfully (range 0-21 prior matches).

In [5]:
from src import manager as mgr
ma = mgr.load_manager_appearances()
fj_matches = mgr.load_matches()
mgr_feats = mgr.manager_features_for_matches(valid, ma=ma, wc_matches=fj_matches)
for c in mgr_feats.columns:
    valid[c] = mgr_feats[c].fillna(0)
valid['mgr_match_diff']   = valid['home_mgr_prior_wc_matches']  - valid['away_mgr_prior_wc_matches']
valid['mgr_winrate_diff'] = valid['home_mgr_prior_wc_win_rate'] - valid['away_mgr_prior_wc_win_rate']
print('WC matches with manager data:', (valid['mgr_match_diff'].abs() > 0).sum())
print(valid[['mgr_match_diff','mgr_winrate_diff']].describe().round(3))

WC matches with manager data: 170
       mgr_match_diff  mgr_winrate_diff
count        7464.000          7464.000
mean            0.029             0.002
std             1.167             0.081
min           -16.000            -1.000
25%             0.000             0.000
50%             0.000             0.000
75%             0.000             0.000
max            17.000             1.000


In [6]:
FEAT_FULL    = FEAT_WITH_FIFA + ['mgr_match_diff','mgr_winrate_diff']
rows2 = []
for year in [2010, 2014, 2018, 2022]:
    train = valid[valid['date'].dt.year < year]
    test  = valid[(valid['date'].dt.year == year) & (valid['tournament'] == 'FIFA World Cup')]
    if test.empty: continue
    y_train = train['outcome'].to_numpy(); y_test = test['outcome'].to_numpy()
    row = {'year': year, 'n_test': len(test)}
    for feat_name, cols in [('with_fifa', FEAT_WITH_FIFA), ('with_fifa+mgr', FEAT_FULL)]:
        sc = StandardScaler().fit(train[cols].to_numpy())
        Xtr = train[cols].to_numpy(); Xte = test[cols].to_numpy()
        Xtr_s = sc.transform(Xtr); Xte_s = sc.transform(Xte)
        for model_name, clf in [
            ('logit', fit_logit(Xtr, y_train)),
            ('xgb',   fit_xgb(Xtr, y_train)),
            ('mlp',   fit_mlp(Xtr_s, y_train)),
        ]:
            X_pred = Xte_s if model_name == 'mlp' else Xte
            p = predict_proba_ordered(clf, X_pred)
            s = score(y_test, p)
            k = f'{model_name}_{feat_name}'
            row[f'{k}_ll']  = s['log_loss']
            row[f'{k}_acc'] = s['accuracy']
            row[f'{k}_br']  = s['brier']
    rows2.append(row)

res = pd.DataFrame(rows2)
print('Averages:')
for name in ['logit_with_fifa','logit_with_fifa+mgr','xgb_with_fifa','xgb_with_fifa+mgr','mlp_with_fifa','mlp_with_fifa+mgr']:
    print(f"  {name:24s}  log_loss: {res[f'{name}_ll'].mean():.4f}   accuracy: {res[f'{name}_acc'].mean():.4f}   brier: {res[f'{name}_br'].mean():.4f}")

Averages:
  logit_with_fifa           log_loss: 1.0096   accuracy: 0.5312   brier: 0.5998
  logit_with_fifa+mgr       log_loss: 1.0291   accuracy: 0.5352   brier: 0.6129
  xgb_with_fifa             log_loss: 1.0079   accuracy: 0.5234   brier: 0.5987
  xgb_with_fifa+mgr         log_loss: 1.0480   accuracy: 0.5156   brier: 0.6187
  mlp_with_fifa             log_loss: 1.0007   accuracy: 0.5469   brier: 0.5954
  mlp_with_fifa+mgr         log_loss: 1.0980   accuracy: 0.5117   brier: 0.6465


In [7]:
# XGBoost gain importance with manager features included
last_train = valid[valid['date'].dt.year < 2022]
clf_xgb = fit_xgb(last_train[FEAT_FULL].to_numpy(), last_train['outcome'].to_numpy())
imp = pd.Series(clf_xgb.feature_importances_, index=FEAT_FULL).sort_values(ascending=False)
print('XGB gain importance (all features):')
print(imp.round(3).to_string())

XGB gain importance (all features):
elo_diff_norm        0.316
fifa_points_diff     0.204
log_value_diff       0.151
top1_share_diff      0.095
outfield_age_diff    0.089
mgr_winrate_diff     0.073
mgr_match_diff       0.072


## WC-only training: does manager feature help when every training row has it?

In [8]:
# Walk-forward but train ONLY on prior WC matches (so every training row has
# manager data, not just 3% of them).
rows3 = []
for year in [2014, 2018, 2022]:  # 2010 has no prior WC training set
    train = valid[(valid['tournament']=='FIFA World Cup') & (valid['date'].dt.year < year) & (valid['date'].dt.year >= 2010)]
    test  = valid[(valid['tournament']=='FIFA World Cup') & (valid['date'].dt.year == year)]
    if test.empty or len(train) < 30: continue
    y_train = train['outcome'].to_numpy(); y_test = test['outcome'].to_numpy()
    row = {'year': year, 'n_train': len(train), 'n_test': len(test)}
    for feat_name, cols in [('no_mgr', FEAT_WITH_FIFA), ('with_mgr', FEAT_FULL)]:
        sc = StandardScaler().fit(train[cols].to_numpy())
        Xtr = train[cols].to_numpy(); Xte = test[cols].to_numpy()
        Xtr_s = sc.transform(Xtr); Xte_s = sc.transform(Xte)
        for model_name, clf in [
            ('logit', fit_logit(Xtr, y_train)),
            ('xgb',   fit_xgb(Xtr, y_train)),
        ]:
            p = predict_proba_ordered(clf, Xte)
            s = score(y_test, p)
            row[f'{model_name}_{feat_name}_ll'] = s['log_loss']
            row[f'{model_name}_{feat_name}_acc'] = s['accuracy']
    rows3.append(row)
res3 = pd.DataFrame(rows3)
print('WC-only training:')
print(res3.round(4).to_string(index=False))
print()
print('Averages:')
for name in ['logit_no_mgr','logit_with_mgr','xgb_no_mgr','xgb_with_mgr']:
    print(f"  {name:18s}  log_loss: {res3[f'{name}_ll'].mean():.4f}   accuracy: {res3[f'{name}_acc'].mean():.4f}")

WC-only training:
 year  n_train  n_test  logit_no_mgr_ll  logit_no_mgr_acc  xgb_no_mgr_ll  xgb_no_mgr_acc  logit_with_mgr_ll  logit_with_mgr_acc  xgb_with_mgr_ll  xgb_with_mgr_acc
 2014       64      64           0.9904            0.5781         1.3924          0.5156             1.0431              0.5156           1.3031            0.5469
 2018      128      64           0.9938            0.5469         1.6801          0.4062             1.0209              0.5156           1.6259            0.4688
 2022      192      64           1.0263            0.5781         1.4323          0.4844             1.0697              0.5469           1.4567            0.5000

Averages:
  logit_no_mgr        log_loss: 1.0035   accuracy: 0.5677
  logit_with_mgr      log_loss: 1.0446   accuracy: 0.5260
  xgb_no_mgr          log_loss: 1.5016   accuracy: 0.4688
  xgb_with_mgr        log_loss: 1.4619   accuracy: 0.5052


## Binary `is_new_manager` feature

User hypothesis: brand-new managers (no prior WC matches) should be a negative shock. Descriptive stats showed the effect only exists for lower-tier teams (n=5 experienced lower-tier mgrs — thin), with new top-tier managers actually slightly outperforming. So a flat `is_new` feature is unlikely to help logit. Test if XGBoost can find the interaction.

In [9]:
valid['home_is_new_mgr'] = (valid['home_mgr_prior_wc_matches'] == 0).astype(int)
valid['away_is_new_mgr'] = (valid['away_mgr_prior_wc_matches'] == 0).astype(int)
# Only meaningful where we have manager data; otherwise 0/0 noise as before.
has_mgr = (valid['home_mgr_prior_wc_matches'] + valid['away_mgr_prior_wc_matches'] > 0) | (valid['tournament'] == 'FIFA World Cup')
valid['is_new_diff'] = (valid['home_is_new_mgr'] - valid['away_is_new_mgr']) * has_mgr.astype(int)

FEAT_NEW = FEAT_WITH_FIFA + ['is_new_diff']
rows4 = []
for year in [2010, 2014, 2018, 2022]:
    train = valid[valid['date'].dt.year < year]
    test  = valid[(valid['date'].dt.year == year) & (valid['tournament'] == 'FIFA World Cup')]
    if test.empty: continue
    y_train = train['outcome'].to_numpy(); y_test = test['outcome'].to_numpy()
    row = {'year': year}
    for feat_name, cols in [('baseline', FEAT_WITH_FIFA), ('+is_new', FEAT_NEW)]:
        for model_name, clf in [
            ('logit', fit_logit(train[cols].to_numpy(), y_train)),
            ('xgb',   fit_xgb(train[cols].to_numpy(), y_train)),
        ]:
            p = predict_proba_ordered(clf, test[cols].to_numpy())
            s = score(y_test, p)
            k = f'{model_name}_{feat_name}'
            row[f'{k}_ll']  = s['log_loss']
            row[f'{k}_acc'] = s['accuracy']
    rows4.append(row)
res4 = pd.DataFrame(rows4)
print('Averages:')
for name in ['logit_baseline','logit_+is_new','xgb_baseline','xgb_+is_new']:
    print(f"  {name:18s}  log_loss: {res4[f'{name}_ll'].mean():.4f}   accuracy: {res4[f'{name}_acc'].mean():.4f}")

Averages:
  logit_baseline      log_loss: 1.0096   accuracy: 0.5312
  logit_+is_new       log_loss: 1.0198   accuracy: 0.5156
  xgb_baseline        log_loss: 1.0079   accuracy: 0.5234
  xgb_+is_new         log_loss: 1.0332   accuracy: 0.5312
